### SILVER NOTOBOOK

This notebook creates the Silver layer of the Amazon sales dataset in the Lakehouse. 
It loads the raw data from the Bronze layer into a Spark DataFrame and performs cleaning and standardization to ensure consistency for analysis. 
ID columns are cleaned of alphabetical prefixes, numeric and financial columns are cast to appropriate types, text fields are trimmed, and dates are converted to proper Date type. 
Finally, the cleaned dataset is saved as a Delta table in the Silver layer, providing a reliable structured source for downstream analytics and star schema modeling.

Import PySpark functions and data types that will be used later for data cleaning and transformations.
These functions allow operations such as selecting columns, replacing characters with regular expressions,
formatting numbers, concatenating strings, converting values to dates, trimming whitespace, and counting records.
Data types are also imported to properly cast columns during the data preparation process.

In [1]:
from pyspark.sql.functions import col, regexp_replace, format_number, concat, lit, to_date, trim, count
from pyspark.sql.types import IntegerType, DecimalType

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 3, Finished, Available, Finished, False)

Load the raw dataset from the Bronze layer of the Lakehouse.
This table contains the original Amazon sales data before any transformation.
The dataset is read into a Spark DataFrame to allow further cleaning,
validation, and preparation for the analytical model.
The display() function is used to visually inspect the data structure and contents.

In [2]:
df = spark.read.table("lakehouse_bronze.DataFrame.amazon")
display(df)

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 426d8ffe-69a8-492a-91a2-cbb088424bf9)

Check for duplicate OrderID values in the dataset.
The data is grouped by OrderID and the number of records for each OrderID is counted.
If the count is greater than 1, it means that the OrderID appears multiple times.
This step helps identify potential data quality issues before proceeding
with further transformations or modeling.

In [4]:
duplicates_df = df.groupBy("OrderID").agg(count("*").alias("count")) \
    .filter(col("count") > 1)

num_duplicates = duplicates_df.count()
print(f"OrderID duplicate = {num_duplicates}")

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 6, Finished, Available, Finished, False)

Order ID duplicate = 0


Clean identifier columns by removing alphabetical prefixes.
Some ID fields in the dataset contain letter prefixes (e.g., "ORD12345", "CUS6789").
Using a regular expression, the code removes any letters at the beginning of the string,
keeping only the numeric part of the identifier.
This helps standardize the ID format and prepares the columns for consistent use
in joins, modeling, and potential type casting.

In [5]:
df = df.withColumn("OrderID", regexp_replace(col("OrderID"), "^[A-Z]+", "")) \
       .withColumn("CustomerID", regexp_replace(col("CustomerID"), "^[A-Z]+", "")) \
       .withColumn("ProductID", regexp_replace(col("ProductID"), "^[A-Z]+", "")) \
       .withColumn("SellerID", regexp_replace(col("SellerID"), "^[A-Z]+", ""))

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 7, Finished, Available, Finished, False)

Convert the Quantity column to Integer type.
In the raw dataset, numeric fields may be stored as strings.
Casting the column ensures that Quantity is treated as a numeric value,
allowing correct calculations, aggregations, and analysis later in the pipeline.

In [6]:
df = df.withColumn("Quantity", col("Quantity").cast(IntegerType()))

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 8, Finished, Available, Finished, False)

Convert monetary columns to Decimal type with two decimal places.
Financial values in the raw dataset may be stored as strings or generic numeric types.
Casting them to Decimal(10,2) ensures consistent precision and accuracy
for calculations such as totals, averages, and financial analysis.
This step prepares the data for reliable aggregation and reporting.

In [7]:
df = df.withColumn("UnitPrice", col("UnitPrice").cast(DecimalType(10,2))) \
       .withColumn("Discount", col("Discount").cast(DecimalType(10,2))) \
       .withColumn("Tax", col("Tax").cast(DecimalType(10,2))) \
       .withColumn("ShippingCost", col("ShippingCost").cast(DecimalType(10,2))) \
       .withColumn("TotalAmount", col("TotalAmount").cast(DecimalType(10,2)))

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 9, Finished, Available, Finished, False)

Convert the OrderDate column to a proper Date type.
The raw dataset stores dates as strings, so they must be parsed
using the correct format ("yyyy-MM-dd").
Converting this column ensures that the field can be used for
time-based analysis such as filtering, grouping by month/year,
and building a calendar dimension for the star schema.

In [8]:
df = df.withColumn(
    "OrderDate",
    to_date(col("OrderDate"), "yyyy-MM-dd"))

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 10, Finished, Available, Finished, False)

Remove leading and trailing whitespace from text columns.
In raw datasets, text fields often contain extra spaces that can cause
inconsistencies when grouping, filtering, or joining data.
The trim() function ensures that all text values are standardized
by removing unnecessary spaces from the beginning and end of each value.

In [9]:
cols_text = ["CustomerName", "ProductName", "Category", "Brand", "PaymentMethod", "OrderStatus", "City", "State", "Country"]

for c in cols_text:
    df = df.withColumn(c, trim(col(c)))

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 11, Finished, Available, Finished, False)

In [10]:
display(df)

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9cdc129f-1b1c-40e5-a835-f799eb071a79)

Save the cleaned dataset into the Silver layer of the Lakehouse.
At this stage, the data has been standardized (IDs cleaned, data types corrected,
text fields trimmed, and dates properly formatted).
The dataset is written as a Delta table, which supports efficient storage,
schema management, and reliable querying for downstream analytics.
The overwrite mode ensures that the table is replaced with the latest
processed version of the data.

In [11]:
df.write.option("overwriteSchema", "true").format("delta").mode("overwrite").saveAsTable("lakehouse_silver.DataFrame.Amazon")

StatementMeta(, a9b9498b-02cb-4429-ac98-30977fb28bee, 13, Finished, Available, Finished, False)